# AI-Powered Sentiment Analysis System
**Month 2 Project | NLP + Machine Learning**

Objective: classify customer review text as Positive, Negative, or Neutral.

This notebook covers data understanding, cleaning, EDA, text representation with TF-IDF, model comparison, evaluation, cross-validation, Grid Search, and serialization.

**Dataset caution:** the 10,000 rows contain only 300 unique `review_text` strings. Therefore, a normal random row split would put identical text templates into both training and testing data and could make evaluation misleading. This notebook keeps identical review texts together during splitting and grouped cross-validation. This is an important data-quality finding to discuss in the report.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

## 2. Load data

In [ ]:
# In Colab, upload the CSV or update DATA_PATH to your Drive location.
DATA_PATH = Path("../data/reviews.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 3. Data quality and repeated-template audit

In [ ]:
display(df.isna().sum().rename("missing_values").to_frame())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate review IDs:", df["review_id"].duplicated().sum())
print("Unique review texts:", df["review_text"].nunique())
print("Repeated rows beyond unique texts:", len(df) - df["review_text"].nunique())
print("Number of text strings linked to multiple sentiment labels:",
      int((df.groupby("review_text")["sentiment"].nunique() > 1).sum()))
display(df.groupby("sentiment")["review_text"].nunique().rename("unique_texts_per_class").to_frame())

## 4. EDA

In [ ]:
display(df["sentiment"].value_counts().rename("count").to_frame())
display((df["sentiment"].value_counts(normalize=True)*100).round(2).rename("percent").to_frame())
ax = df["sentiment"].value_counts().reindex(["Positive","Negative","Neutral"]).plot(kind="bar")
ax.set_title("Sentiment Class Distribution"); ax.set_xlabel("Sentiment"); ax.set_ylabel("Rows")
plt.tight_layout(); plt.show()
display(df["category"].value_counts().to_frame("reviews"))
display(df["source"].value_counts().to_frame("reviews"))

## 5. Prepare text and target

Use `review_text` as the only input and `sentiment` as the target. TF-IDF performs tokenization and creates numeric features within the pipeline. We do not blindly remove stop words: negation terms such as “not” can carry sentiment meaning.

In [ ]:
df = df.dropna(subset=["sentiment"]).copy()
df["review_text"] = df["review_text"].fillna("").astype(str).str.strip()
df = df[df["review_text"].ne("")]
X = df["review_text"]
y = df["sentiment"].astype(str)
groups = X  # use exact review text as the group key
print("Rows:", len(df), "| Unique text templates:", X.nunique())

## 6. Group-aware train/test split

In [ ]:
# Identical review text templates cannot appear in both sets.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
assert set(X_train).isdisjoint(set(X_test))
print("Train rows:", len(X_train), "| train unique texts:", X_train.nunique())
print("Test rows:", len(X_test), "| test unique texts:", X_test.nunique())
print("Train/test text overlap:", len(set(X_train) & set(X_test)))

## 7. Baseline model comparison

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=30000, sublinear_tf=True)),
        ("classifier", LogisticRegression(max_iter=1500, random_state=42))]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=30000)),
        ("classifier", MultinomialNB())]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=30000, sublinear_tf=True)),
        ("classifier", LinearSVC(random_state=42))])
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test,p),
                 "Precision (macro)": precision_score(y_test,p,average="macro",zero_division=0),
                 "Recall (macro)": recall_score(y_test,p,average="macro",zero_division=0),
                 "F1-score (macro)": f1_score(y_test,p,average="macro",zero_division=0)})
comparison = pd.DataFrame(rows).sort_values("F1-score (macro)",ascending=False)
display(comparison)

## 8. Compare metrics visually

In [ ]:
ax = comparison.set_index("Model")[["Accuracy","Precision (macro)","Recall (macro)","F1-score (macro)"]].plot(kind="bar",figsize=(9,5))
ax.set_title("Baseline Model Comparison"); ax.set_ylabel("Score"); ax.set_ylim(0,1.05)
plt.xticks(rotation=15,ha="right"); plt.tight_layout(); plt.show()

## 9. Grouped cross-validation and Grid Search

In [ ]:
tuning_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=30000, sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1500, random_state=42))
])
param_grid = {"tfidf__ngram_range": [(1,1),(1,2)], "classifier__C": [0.5,1.0,2.0]}
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(tuning_pipeline, param_grid, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True)
grid.fit(X_train, y_train, groups=X_train)
print("Best parameters:", grid.best_params_)
print("Best grouped CV macro F1:", round(grid.best_score_,4))
best_model = grid.best_estimator_

## 10. Final held-out evaluation

In [ ]:
y_pred = best_model.predict(X_test)
metrics = {
    "Accuracy": accuracy_score(y_test,y_pred),
    "Precision (macro)": precision_score(y_test,y_pred,average="macro",zero_division=0),
    "Recall (macro)": recall_score(y_test,y_pred,average="macro",zero_division=0),
    "F1-score (macro)": f1_score(y_test,y_pred,average="macro",zero_division=0)
}
display(pd.DataFrame([metrics]).T.rename(columns={0:"Score"}))
print(classification_report(y_test,y_pred,zero_division=0))

## 11. Confusion matrix

In [ ]:
labels = ["Negative","Neutral","Positive"]
cm = confusion_matrix(y_test,y_pred,labels=labels)
ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=labels).plot(values_format="d")
plt.title("Confusion Matrix — Tuned Logistic Regression")
plt.tight_layout(); plt.show()

## 12. Save a reusable model pipeline

In [ ]:
# Refit the selected configuration on all available rows for the application.
final_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=30000, sublinear_tf=True,
                              ngram_range=grid.best_params_["tfidf__ngram_range"])),
    ("classifier", LogisticRegression(max_iter=1500, random_state=42,
                                      C=grid.best_params_["classifier__C"]))
])
final_model.fit(X,y)
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True,exist_ok=True)
joblib.dump(final_model,MODEL_DIR/"sentiment_pipeline.pkl")
print("Saved:", MODEL_DIR/"sentiment_pipeline.pkl")

## 13. Test with example reviews

In [ ]:
examples = [
    "The product is excellent and works exactly as expected.",
    "The item broke quickly and customer service was unhelpful.",
    "The package includes a cable and a charging adapter."
]
for text, label in zip(examples, final_model.predict(examples)):
    print(f"{label:>8} | {text}")

## 14. Conclusion and limitations

- Use the confusion matrix and class-wise scores to examine errors.
- The grouped split avoids the simplest form of leakage from identical text templates appearing in both train and test sets.
- However, only 300 unique text templates are present in the 10,000-row dataset. A perfect score may still reflect highly repetitive, synthetic, or easily separable examples rather than real-world robustness.
- Test on genuinely new reviews from independent sources before claiming real-world performance.
- Accuracy, precision, recall, and F1 describe this evaluation setup; they are not a guarantee of future performance.